# how it works 

- Encoder (LSTM) → Understands input text
- Attention → Focus on important words
- Decoder (LSTM) → Generates summary

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_from_disk

In [2]:
dataset = load_from_disk("summarizer_dataset")

dataset.set_format(type="torch", columns=["input_ids", "labels"])

loader = DataLoader(dataset, batch_size=8, shuffle=True)

In [3]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        self.lstm = nn.LSTM(
            embed_size, 
            hidden_size, 
            num_layers=2,   #  2-layer stacked LSTM
            batch_first=True
        )

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

In [4]:
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, hidden, encoder_outputs):
        # hidden: (num_layers, batch, hidden)
        hidden = hidden[-1].unsqueeze(2)  # last layer
        
        # Compute scores
        scores = torch.bmm(encoder_outputs, hidden).squeeze(2)
        
        attn_weights = self.softmax(scores)
        
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        
        return context

In [5]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size + hidden_size, hidden_size, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden, cell, context):
        embedded = self.embedding(x)
        
        # Combine context + input
        context = context.repeat(1, embedded.size(1), 1)
        input_combined = torch.cat((embedded, context), dim=2)
        
        outputs, (hidden, cell) = self.lstm(input_combined, (hidden, cell))
        
        predictions = self.fc(outputs)
        
        return predictions, hidden, cell

In [6]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, attention):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.attention = attention

    def forward(self, src, trg):
        encoder_outputs, hidden, cell = self.encoder(src)
        
        context = self.attention(hidden, encoder_outputs)
        
        outputs, _, _ = self.decoder(trg, hidden, cell, context)
        
        return outputs

In [7]:
vocab_size = 32128
embed_size = 128
hidden_size = 512

encoder = Encoder(vocab_size, embed_size, hidden_size)
decoder = Decoder(vocab_size, embed_size, hidden_size)
attention = Attention(hidden_size)

model = Seq2Seq(encoder, decoder, attention)

In [10]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.load_state_dict(torch.load("lstm_model.pth", map_location=device))
model.to(device)
model.eval()

print("Model loaded successfully!")

Model loaded successfully!


In [12]:
import torch.nn.utils as utils

for epoch in range(2):
    total_loss = 0
    
    for batch in loader:
        src = batch["input_ids"].to(device)
        trg = batch["labels"].to(device)
        
        optimizer.zero_grad()
        
        output = model(src, trg)
        
        loss = criterion(
            output.view(-1, vocab_size),
            trg.view(-1)
        )
        
        loss.backward()
        
        # Gradient clipping
        utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.2f}")


Epoch 1, Loss: 453.29
Epoch 2, Loss: 70.28


In [13]:
print(loss.item())

0.08900830894708633


In [ ]:
torch.save(model.state_dict(), "lstm_model.pth")
print("Model saved successfully!")

Model saved successfully!


In [ ]:
print(type(model))

<class '__main__.Seq2Seq'>


In [14]:
print(model)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(32128, 128)
    (lstm): LSTM(128, 512, num_layers=2, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(32128, 128)
    (lstm): LSTM(640, 512, num_layers=2, batch_first=True)
    (fc): Linear(in_features=512, out_features=32128, bias=True)
  )
  (attention): Attention(
    (softmax): Softmax(dim=1)
  )
)


# lstm_evaluation

In [16]:
model.eval()

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(32128, 128)
    (lstm): LSTM(128, 512, num_layers=2, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(32128, 128)
    (lstm): LSTM(640, 512, num_layers=2, batch_first=True)
    (fc): Linear(in_features=512, out_features=32128, bias=True)
  )
  (attention): Attention(
    (softmax): Softmax(dim=1)
  )
)

In [17]:
def generate_summary(model, input_ids, max_len=50):
    model.eval()
    
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(input_ids)
        context = model.attention(hidden, encoder_outputs)
        
        decoder_input = input_ids[:, :1]
        outputs = []
        
        for _ in range(max_len):
            preds, hidden, cell = model.decoder(
                decoder_input, hidden, cell, context
            )
            
            next_token = preds[:, -1, :].argmax(dim=-1, keepdim=True)
            outputs.append(next_token)
            decoder_input = next_token
        
        return torch.cat(outputs, dim=1)

In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("t5-small")

sample = dataset[0]

input_ids = sample["input_ids"].unsqueeze(0)
target_ids = sample["labels"]

generated_ids = generate_summary(model, input_ids)

print("Generated Summary:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

print("\nActual Summary:")
print(tokenizer.decode(target_ids, skip_special_tokens=True))

Generated Summary:
L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L L

Actual Summary:
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday. Young actor says he has no plans to fritter his cash away. Radcliffe's earnings from first five Potter films have been held in trust fund.


In [2]:
#LSTM models struggle with long text summarization due to limited memory and exposure bias, 
# which results in repetitive outputs. This highlights the need for transformer-based models
# it happens because LSTMs have a fixed-size hidden state that can only capture limited context, leading to poor performance on long sequences. Transformers, with their attention mechanisms, can capture long-range dependencies more effectively, making them better suited for summarization tasks.

In [21]:
!pip install rouge-score

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=25027 sha256=51625c642d4368e27c311a9004f89701133246b75d6545609ed49f2ca191fdfb
  Stored in directory: c:\users\harsh\appdata\local\pip\cache\wheels\1e\19\43\8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [22]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

generated = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
actual = tokenizer.decode(target_ids, skip_special_tokens=True)

scores = scorer.score(actual, generated)

print(scores)

{'rouge1': Score(precision=0.0, recall=0.0, fmeasure=0.0), 'rouge2': Score(precision=0.0, recall=0.0, fmeasure=0.0), 'rougeL': Score(precision=0.0, recall=0.0, fmeasure=0.0)}


In [1]:
#The LSTM model failed because it struggles with long-range dependencies and generated repetitive, meaningless tokens, resulting in poor summarization performance.

#Transformer models are needed as they use attention mechanisms to better understand context and generate accurate, coherent summaries.